# TabICLRegressor device benchmark

Compare **CPU** vs available accelerators (**CUDA / XPU / MPS**) for `TabICLRegressor`:
timing, host RAM, device memory, prediction parity, and R² — with and without `kv_cache`.

Dataset: `sklearn.datasets.make_friedman1` (2000×50, 50/50 split, noise tuned so a strong tree model sits near R²≈0.85).

In [1]:
from __future__ import annotations

import platform
import subprocess
import threading
import time
from dataclasses import asdict, dataclass
from importlib.metadata import PackageNotFoundError, version
import gc

import numpy as np
import pandas as pd
import psutil
import torch
from sklearn.datasets import make_friedman1
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

import tabicl
from tabicl import TabICLRegressor

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:.4g}")

## Software versions

In [2]:
def pkg_version(name: str) -> str:
    try:
        return version(name)
    except PackageNotFoundError:
        return "not installed"


versions = {
    "tabicl": getattr(tabicl, "__version__", pkg_version("tabicl")),
    "torch": torch.__version__,
    "numpy": np.__version__,
    "scikit-learn": pkg_version("scikit-learn"),
    "scipy": pkg_version("scipy"),
    "einops": pkg_version("einops"),
    "psutil": pkg_version("psutil"),
    "huggingface-hub": pkg_version("huggingface-hub"),
    "pandas": pd.__version__,
    "python": platform.python_version(),
}
pd.DataFrame({"package": list(versions), "version": list(versions.values())})

,package,version
0,tabicl,2.2.0
1,torch,2.14.1+xpu
2,numpy,2.5.3
3,scikit-learn,1.9.1
4,scipy,1.18.1
5,einops,0.8.2
6,psutil,7.2.2
7,huggingface-hub,1.32.0
8,pandas,3.0.6
9,python,3.13.15


## OS, CPU, RAM, and accelerators

In [3]:
def sysctl(name: str) -> str | None:
    if platform.system() != "Darwin":
        return None
    try:
        return subprocess.check_output(["sysctl", "-n", name], text=True, stderr=subprocess.DEVNULL).strip()
    except (OSError, subprocess.CalledProcessError):
        return None


def backend_available(name: str) -> bool:
    api = getattr(torch, name, None)
    is_available = getattr(api, "is_available", None)
    return callable(is_available) and bool(is_available())


def device_memory_bytes(device: torch.device) -> tuple[int | None, int | None]:
    """Return (allocated, total) device memory in bytes when the backend exposes it."""
    api = getattr(torch, device.type, None)
    if api is None:
        return None, None

    synchronize = getattr(api, "synchronize", None)
    if callable(synchronize):
        try:
            synchronize()
        except Exception:
            pass

    mem_get_info = getattr(api, "mem_get_info", None)
    if callable(mem_get_info):
        try:
            free, total = mem_get_info(device)
        except TypeError:
            free, total = mem_get_info()
        return total - free, total

    # MPS-style APIs
    current = getattr(api, "current_allocated_memory", None)
    recommended = getattr(api, "recommended_max_memory", None)
    allocated = current() if callable(current) else None
    total = recommended() if callable(recommended) else None
    return allocated, total


def device_name(device_type: str) -> str | None:
    if device_type == "cuda" and torch.cuda.is_available():
        return torch.cuda.get_device_name(0)
    if device_type == "xpu" and backend_available("xpu"):
        get_name = getattr(torch.xpu, "get_device_name", None)
        return get_name(0) if callable(get_name) else "xpu:0"
    if device_type == "mps" and backend_available("mps"):
        return sysctl("machdep.cpu.brand_string") or "Apple Metal (MPS)"
    return None


cpu_brand = (
    sysctl("machdep.cpu.brand_string")
    or platform.processor()
    or platform.machine()
)
vm = psutil.virtual_memory()
swap = psutil.swap_memory()

host_info = {
    "os": f"{platform.system()} {platform.release()}",
    "platform": platform.platform(),
    "cpu_arch": platform.machine(),
    "cpu_name": cpu_brand,
    "cpu_logical_cores": psutil.cpu_count(logical=True),
    "cpu_physical_cores": psutil.cpu_count(logical=False),
    "ram_total_gb": vm.total / 1e9,
    "ram_available_gb": vm.available / 1e9,
    "swap_total_gb": swap.total / 1e9,
    "swap_used_gb": swap.used / 1e9,
}
pd.Series(host_info).to_frame("value")

,value
os,Linux 7.2.7-arch1-1
platform,Linux-7.2.7-arch1-1-x86_64-with-glibc2.44
cpu_arch,x86_64
cpu_name,x86_64
cpu_logical_cores,8
cpu_physical_cores,8
ram_total_gb,33.12
ram_available_gb,9.804
swap_total_gb,4.295
swap_used_gb,2.007


In [4]:
accelerators = []
for name in ("cuda", "xpu", "mps"):
    available = backend_available(name)
    row = {
        "device": name,
        "available": available,
        "name": device_name(name) if available else None,
        "allocated_gb": None,
        "total_gb": None,
        "available_gb": None,
        "runtime": None,
    }
    if available:
        allocated, total = device_memory_bytes(torch.device(name))
        if allocated is not None:
            row["allocated_gb"] = allocated / 1e9
        if total is not None:
            row["total_gb"] = total / 1e9
            if allocated is not None:
                row["available_gb"] = (total - allocated) / 1e9
        if name == "cuda":
            row["runtime"] = f"CUDA {torch.version.cuda}"
        elif name == "xpu":
            row["runtime"] = "Intel XPU"
        elif name == "mps":
            row["runtime"] = "Apple Metal Performance Shaders"
    accelerators.append(row)

accel_df = pd.DataFrame(accelerators)
accel_df

,device,available,name,allocated_gb,total_gb,available_gb,runtime
0,cuda,False,NaN,NaN,NaN,NaN,NaN
1,xpu,True,Intel(R) Graphics,25.42,30.66,5.246,Intel XPU
2,mps,False,NaN,NaN,NaN,NaN,NaN


## Dataset (`make_friedman1`)

Noise is set so `HistGradientBoostingRegressor` lands near R²≈0.85 on the held-out half (a proxy for a strong tabular baseline).

In [5]:
N_SAMPLES = 2000
N_FEATURES = 50
NOISE = 1.5
RANDOM_STATE = 0
N_ESTIMATORS = 8  # TabICL ensemble size

X, y = make_friedman1(
    n_samples=N_SAMPLES,
    n_features=N_FEATURES,
    noise=NOISE,
    random_state=RANDOM_STATE,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, random_state=RANDOM_STATE
)

hgb = HistGradientBoostingRegressor(random_state=RANDOM_STATE)
hgb.fit(X_train, y_train)
hgb_r2 = r2_score(y_test, hgb.predict(X_test))

pd.Series(
    {
        "n_train": len(X_train),
        "n_test": len(X_test),
        "n_features": X.shape[1],
        "noise": NOISE,
        "y_train_std": float(np.std(y_train)),
        "hgb_test_r2": hgb_r2,
    }
).to_frame("value")

,value
n_train,1000
n_test,1000
n_features,50
noise,1.5
y_train_std,5.114
hgb_test_r2,0.8472


## Benchmark helpers

Host peak RSS is sampled on a background thread. Device peak memory uses backend peak counters when available (CUDA/XPU); on MPS we sample `current_allocated_memory` during the run.

In [6]:
@dataclass
class RunResult:
    label: str
    device: str
    use_amp: bool
    kv_cache: bool
    fit_s: float
    predict_s: float
    total_s: float
    speedup_vs_cpu: float | None
    r2: float
    r2_rel_degradation_vs_cpu: float | None
    mean_abs_diff_vs_cpu: float | None
    max_abs_diff_vs_cpu: float | None
    host_rss_peak_increment_gb: float
    device_mem_peak_increment_gb: float
    swap_used_before_gb: float
    swap_used_after_gb: float
    swap_grew: bool


def reset_device_peak_stats(device: torch.device) -> None:
    api = getattr(torch, device.type, None)
    if api is None:
        return
    reset = getattr(api, "reset_peak_memory_stats", None)
    if callable(reset):
        try:
            reset(device)
        except TypeError:
            reset()
        except Exception:
            pass
    empty_cache = getattr(api, "empty_cache", None)
    if callable(empty_cache):
        try:
            empty_cache()
        except Exception:
            pass

def synchronize_device(device: torch.device) -> None:
    api = getattr(torch, device.type, None)
    synchronize = getattr(api, "synchronize", None) if api is not None else None
    if callable(synchronize):
        try:
            synchronize()
        except Exception:
            pass


class MemorySampler:
    """Sample host RSS and optional device-allocated bytes while a callable runs."""

    def __init__(self, device: torch.device | None = None, interval_s: float = 0.02, verbose: bool = False):
        self.device = device
        self.interval_s = interval_s
        self.proc = psutil.Process()
        self.peak_rss = 0
        self.peak_device = 0
        self._stop = threading.Event()
        self._thread: threading.Thread | None = None
        self.verbose = verbose

    def _device_allocated(self) -> int:
        if self.device is None or self.device.type == "cpu":
            return 0
        api = getattr(torch, self.device.type, None)
        if api is None:
            return 0
        # Prefer allocator peak when exposed
        max_alloc = getattr(api, "max_memory_allocated", None)
        if callable(max_alloc):
            try:
                return int(max_alloc(self.device))
            except TypeError:
                return int(max_alloc())
            except Exception:
                pass
        current = getattr(api, "current_allocated_memory", None)
        if callable(current):
            try:
                return int(current())
            except Exception:
                return 0
        memory_allocated = getattr(api, "memory_allocated", None)
        if callable(memory_allocated):
            try:
                return int(memory_allocated(self.device))
            except TypeError:
                return int(memory_allocated())
            except Exception:
                return 0
        return 0

    def _loop(self) -> None:
        i = 0
        while not self._stop.is_set():
            self.rss_peak = max(self.rss_peak, self.proc.memory_info().rss)
            self.device_peak = max(self.device_peak, self._device_allocated())
            i += 1
            if self.verbose and i % 50 == 0:
                print(f"PeakSampler: rss_peak={self.rss_peak / 1e9:.3f} GB, device_peak={self.device_peak / 1e9:.3f} GB")
            time.sleep(self.interval_s)
            

    def __enter__(self):
        # Encourage the allocator to release leftovers from the previous run.
        reset_device_peak_stats(self.device)
        synchronize_device(self.device)
        gc.collect()
        self.rss_peak = self.rss_before = self.proc.memory_info().rss
        self.device_peak = self.device_before = self._device_allocated()
        self._stop.clear()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, *exc):
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=2.0)
        self.rss_peak = max(self.proc.memory_info().rss, self.rss_peak)
        self.device_peak = max(self._device_allocated(), self.device_peak)


def run_tabicl(
    *,
    device: str,
    use_amp: bool,
    kv_cache: bool,
    label: str,
    cpu_pred: np.ndarray | None = None,
    cpu_total_s: float | None = None,
    cpu_r2: float | None = None,
) -> tuple[RunResult, np.ndarray]:
    torch_device = torch.device(device)

    reg = TabICLRegressor(
        n_estimators=N_ESTIMATORS,
        device=device,
        use_amp=use_amp,
        use_fa3=False,
        kv_cache=kv_cache,
        random_state=RANDOM_STATE,
        verbose=False,
    )
    swap_before = psutil.swap_memory().used

    with MemorySampler(device=torch_device) as mem_sampler:
        t0 = time.perf_counter()
        reg.fit(X_train, y_train)
        synchronize_device(torch_device)
        t1 = time.perf_counter()
        pred = reg.predict(X_test)
        synchronize_device(torch_device)
        t2 = time.perf_counter()

    swap_after = psutil.swap_memory().used
    fit_s = t1 - t0
    predict_s = t2 - t1
    total_s = t2 - t0
    r2 = float(r2_score(y_test, pred))

    mean_abs = max_abs = None
    if cpu_pred is not None:
        abs_diff = np.abs(pred - cpu_pred)
        mean_abs = float(abs_diff.mean())
        max_abs = float(abs_diff.max())

    speedup = (cpu_total_s / total_s) if cpu_total_s else None
    r2_deg = None
    if cpu_r2 is not None and abs(cpu_r2) > 1e-12:
        r2_deg = float((cpu_r2 - r2) / abs(cpu_r2))

    host_rss_peak_inc = max(0.0, (mem_sampler.rss_peak - mem_sampler.rss_before) / 1e9)
    device_peak_inc = max(0.0, (mem_sampler.device_peak - mem_sampler.device_before) / 1e9)
    result = RunResult(
        label=label,
        device=device,
        use_amp=use_amp,
        kv_cache=kv_cache,
        fit_s=fit_s,
        predict_s=predict_s,
        total_s=total_s,
        speedup_vs_cpu=speedup,
        r2=r2,
        r2_rel_degradation_vs_cpu=r2_deg,
        mean_abs_diff_vs_cpu=mean_abs,
        max_abs_diff_vs_cpu=max_abs,
        host_rss_peak_increment_gb=host_rss_peak_inc,
        device_mem_peak_increment_gb=device_peak_inc,
        swap_used_before_gb=swap_before / 1e9,
        swap_used_after_gb=swap_after / 1e9,
        # macOS swap accounting can jitter by a few MB from other processes;
        # only flag material growth that could indicate our run paged.
        swap_grew=swap_after > swap_before + 64 * 1e6,
    )
    return result, np.asarray(pred)


available_accelerators = [r["device"] for r in accelerators if r["available"]]
print("Accelerators to benchmark:", available_accelerators or "(none — CPU only)")


Accelerators to benchmark: ['xpu']


## Runs

For each `kv_cache` in `{False, True}`:

1. CPU reference (`use_amp=False`)
2. Each available accelerator with AMP off and on

Accelerator speedups / prediction diffs / R² degradation are relative to the CPU
reference with the **same** `kv_cache` setting.

In [7]:
results: list[RunResult] = []
cpu_refs: dict[bool, tuple[RunResult, np.ndarray]] = {}

for kv_cache in (False, True):
    print(f"\n=== kv_cache={kv_cache} ===", flush=True)

    cpu_label = f"cpu / amp=False / kv_cache={kv_cache}"
    print(f"Running {cpu_label} ...", flush=True)
    cpu_result, cpu_pred = run_tabicl(
        device="cpu",
        use_amp=False,
        kv_cache=kv_cache,
        label=cpu_label,
    )
    results.append(cpu_result)
    cpu_refs[kv_cache] = (cpu_result, cpu_pred)
    print(
        f"  CPU done in {cpu_result.total_s:.3f}s "
        f"(fit {cpu_result.fit_s:.3f}s, predict {cpu_result.predict_s:.3f}s), "
        f"R²={cpu_result.r2:.4f}"
    )

    for device in available_accelerators:
        for use_amp in (False, True):
            label = f"{device} / amp={use_amp} / kv_cache={kv_cache}"
            print(f"Running {label} ...", flush=True)
            result, _ = run_tabicl(
                device=device,
                use_amp=use_amp,
                kv_cache=kv_cache,
                label=label,
                cpu_pred=cpu_pred,
                cpu_total_s=cpu_result.total_s,
                cpu_r2=cpu_result.r2,
            )
            results.append(result)
            print(
                f"  {label}: {result.total_s:.3f}s "
                f"({result.speedup_vs_cpu:.2f}× vs CPU), "
                f"R²={result.r2:.4f}, "
                f"max|Δ|={result.max_abs_diff_vs_cpu:.4g}"
            )

results_df = pd.DataFrame([asdict(r) for r in results])
results_df



=== kv_cache=False ===


Running cpu / amp=False / kv_cache=False ...


  CPU done in 9.503s (fit 0.412s, predict 9.091s), R²=0.9041
Running xpu / amp=False / kv_cache=False ...


  xpu / amp=False / kv_cache=False: 5.338s (1.78× vs CPU), R²=0.9041, max|Δ|=1.431e-05
Running xpu / amp=True / kv_cache=False ...


  xpu / amp=True / kv_cache=False: 3.664s (2.59× vs CPU), R²=0.9040, max|Δ|=0.01837

=== kv_cache=True ===


Running cpu / amp=False / kv_cache=True ...


  CPU done in 11.581s (fit 7.154s, predict 4.427s), R²=0.9041
Running xpu / amp=False / kv_cache=True ...


  xpu / amp=False / kv_cache=True: 4.657s (2.49× vs CPU), R²=0.9041, max|Δ|=1.431e-05
Running xpu / amp=True / kv_cache=True ...


  xpu / amp=True / kv_cache=True: 2.252s (5.14× vs CPU), R²=0.9040, max|Δ|=0.01837


,label,device,use_amp,kv_cache,fit_s,predict_s,total_s,speedup_vs_cpu,r2,r2_rel_degradation_vs_cpu,mean_abs_diff_vs_cpu,max_abs_diff_vs_cpu,host_rss_peak_increment_gb,device_mem_peak_increment_gb,swap_used_before_gb,swap_used_after_gb,swap_grew
0,cpu / amp=False / kv_cache=False,cpu,False,False,0.4121,9.091,9.503,NaN,0.9041,NaN,NaN,NaN,2.72,0,2.007,1.993,False
1,xpu / amp=False / kv_cache=False,xpu,False,False,0.3404,4.997,5.338,1.78,0.9041,-9.018e-09,2.614e-06,1.431e-05,2.023,3.169,1.993,2.101,True
2,xpu / amp=True / kv_cache=False,xpu,True,False,0.3751,3.289,3.664,2.594,0.904,1.142e-05,0.002945,0.01837,1.294,0.8726,2.101,2.906,True
3,cpu / amp=False / kv_cache=True,cpu,False,True,7.154,4.427,11.58,NaN,0.9041,NaN,NaN,NaN,2.11,0,2.906,4.206,True
4,xpu / amp=False / kv_cache=True,xpu,False,True,2.848,1.809,4.657,2.487,0.9041,-2.363e-09,2.559e-06,1.431e-05,0.1209,2.209,4.206,4.295,True
5,xpu / amp=True / kv_cache=True,xpu,True,True,1.483,0.7683,2.252,5.144,0.904,1.114e-05,0.002943,0.01837,0.07733,1.117,4.295,4.292,False
